In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
import pickle
import optuna

In [2]:
data = pd.read_excel('07_ODU2_5min_for_IAT_prediction_model.xlsx')

In [3]:
data.shape

(1735, 14)

In [4]:
data.columns

Index(['ST', 'OAT', 'Delta_IAT_OAT', 'Delta_ST_IAT', 'Delta_ST_OAT',
       'High_Pressure', 'Day_sin', 'Day_cos', 'Op_Mode_0', 'Op_Mode_1',
       'Year_sin', 'Year_cos', 'OTW', 'IAT'],
      dtype='object')

In [5]:
data.head()

,ST,OAT,Delta_IAT_OAT,Delta_ST_IAT,Delta_ST_OAT,High_Pressure,Day_sin,Day_cos,Op_Mode_0,Op_Mode_1,Year_sin,Year_cos,OTW,IAT
0,24.664311,-1.059259,25.912786,-0.189216,25.723570,30.103704,0.779884,-0.625923,0,1,-0.179562,0.983747,-1.863094,24.853527
1,24.664311,-0.950000,29.390094,-3.775783,25.614311,30.733333,0.766044,-0.642788,0,1,-0.179503,0.983757,-1.892156,28.440094
2,24.664311,-0.910526,30.093193,-4.518356,25.574837,30.089474,0.751840,-0.659346,0,1,-0.179444,0.983768,-1.864571,29.182667
3,24.664311,-0.800000,29.234894,-3.770583,25.464311,28.137500,0.737277,-0.675590,0,1,-0.179385,0.983779,-1.823869,28.434894
4,24.664311,-0.800000,28.965769,-3.501458,25.464311,29.806250,0.722364,-0.691513,0,1,-0.179327,0.983790,-1.784245,28.165769


In [6]:
# The normalizing the inputs

def normalize_inputs(data):
    ST_min, ST_max = 18, 31
    IAT_min, IAT_max = 15, 35
    OAT_min, OAT_max = -20, 20
    OTW_min, OTW_max = -15, 10
    High_Pressure_min, High_Pressure_max = 5, 35
    
    
    data['ST'] = (data['ST'] - ST_min) / (ST_max - ST_min)
    data['IAT'] = (data['IAT'] - IAT_min) / (IAT_max - IAT_min)
    data['OAT'] = (data['OAT'] - OAT_min) / (OAT_max - OAT_min)
    data['OTW'] = (data['OTW'] - OTW_min) / (OTW_max - OTW_min)
    data['High_Pressure'] = (data['High_Pressure'] - High_Pressure_min) / (High_Pressure_max - High_Pressure_min)
    
    return data

# Denormalizing the predicted outputs

def inverse_normalization_output(outputs):
    IAT_min, IAT_max = 15, 35

    outputs['IAT'] = outputs['IAT'] * (IAT_max - IAT_min) + IAT_min

    return outputs

In [7]:
# Normalizing outputs

data = normalize_inputs(data)

In [8]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
	n_vars = 1 if type(data) is list else data.shape[1]
	df = pd.DataFrame(data)
	cols, names = list(), list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):
		cols.append(df.shift(i))
		names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
		if i == 0:
			names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
		else:
			names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
	# put it all together
	agg = pd.concat(cols, axis=1)
	agg.columns = names
	# drop rows with NaN values
	if dropnan:
		agg.dropna(inplace=True)
	return agg

In [9]:
values = data.values.astype('float32')
reframed = series_to_supervised(values,2, 1)
reframed.columns

Index(['var1(t-2)', 'var2(t-2)', 'var3(t-2)', 'var4(t-2)', 'var5(t-2)',
       'var6(t-2)', 'var7(t-2)', 'var8(t-2)', 'var9(t-2)', 'var10(t-2)',
       'var11(t-2)', 'var12(t-2)', 'var13(t-2)', 'var14(t-2)', 'var1(t-1)',
       'var2(t-1)', 'var3(t-1)', 'var4(t-1)', 'var5(t-1)', 'var6(t-1)',
       'var7(t-1)', 'var8(t-1)', 'var9(t-1)', 'var10(t-1)', 'var11(t-1)',
       'var12(t-1)', 'var13(t-1)', 'var14(t-1)', 'var1(t)', 'var2(t)',
       'var3(t)', 'var4(t)', 'var5(t)', 'var6(t)', 'var7(t)', 'var8(t)',
       'var9(t)', 'var10(t)', 'var11(t)', 'var12(t)', 'var13(t)', 'var14(t)'],
      dtype='object')

In [10]:
reframed.drop(['var3(t)', 'var4(t)'], axis =1, inplace=True)
reframed = reframed.values

In [11]:
features = reframed[:, :-1]
targets = reframed[:, -1]

In [12]:
train_x, test_x, train_y, test_y = train_test_split(features, targets, test_size=0.2, shuffle=True, random_state=123)

In [13]:
import time
start = time.time()

In [14]:
def objective(trial):

    param = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 3000, step=20),
        "n_estimators": trial.suggest_categorical('n_estimators', [10,30, 50, 100, 200, 500]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.001, 0.01, 0.1, 0.3, 0.008,0.009]),
        'max_depth': trial.suggest_categorical('max_depth', [3,5,7,9,11,13,15,17,20]),
        "reg_alpha": trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda':trial.suggest_loguniform('reg_lambda', 0.5, 1.0),
       
    }
    
    
    cv_score_lr = cross_val_score(LGBMRegressor(**param), train_x, train_y, cv=5,scoring='r2')

    mean_r2 = round(sum(cv_score_lr)/len(cv_score_lr), 3)
    

    return mean_r2

In [15]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000)
print('Number of finished trials:', len(study.trials))

[I 2023-02-02 08:54:30,109] A new study created in memory with name: no-name-d6005512-4522-44a6-9394-812f80887548
[I 2023-02-02 08:54:30,388] Trial 0 finished with value: 0.847 and parameters: {'num_leaves': 1380, 'n_estimators': 200, 'learning_rate': 0.3, 'max_depth': 5, 'reg_alpha': 2.3189704722867823, 'reg_lambda': 0.8457828679229006}. Best is trial 0 with value: 0.847.
[I 2023-02-02 08:54:32,255] Trial 1 finished with value: 0.826 and parameters: {'num_leaves': 840, 'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 17, 'reg_alpha': 0.4974487528924167, 'reg_lambda': 0.8305545321942914}. Best is trial 0 with value: 0.847.
[I 2023-02-02 08:54:32,399] Trial 2 finished with value: 0.144 and parameters: {'num_leaves': 180, 'n_estimators': 10, 'learning_rate': 0.01, 'max_depth': 5, 'reg_alpha': 0.001788022515917909, 'reg_lambda': 0.8188752035449411}. Best is trial 0 with value: 0.847.
[I 2023-02-02 08:54:32,689] Trial 3 finished with value: 0.131 and parameters: {'num_leaves': 1880

[I 2023-02-02 08:56:00,830] Trial 30 finished with value: 0.898 and parameters: {'num_leaves': 1300, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 9, 'reg_alpha': 0.01391457865086237, 'reg_lambda': 0.557786419138489}. Best is trial 21 with value: 0.908.
[I 2023-02-02 08:56:04,312] Trial 31 finished with value: 0.908 and parameters: {'num_leaves': 1520, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.050558619464670565, 'reg_lambda': 0.6509671058649182}. Best is trial 21 with value: 0.908.
[I 2023-02-02 08:56:06,781] Trial 32 finished with value: 0.908 and parameters: {'num_leaves': 760, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.0983202517887285, 'reg_lambda': 0.6556294018383606}. Best is trial 21 with value: 0.908.
[I 2023-02-02 08:56:09,706] Trial 33 finished with value: 0.905 and parameters: {'num_leaves': 800, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.0676059446176718, 'reg_lambda': 0

[I 2023-02-02 08:57:33,725] Trial 60 finished with value: 0.812 and parameters: {'num_leaves': 920, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 4.675432483799909, 'reg_lambda': 0.6455385242315521}. Best is trial 51 with value: 0.909.
[I 2023-02-02 08:57:38,257] Trial 61 finished with value: 0.906 and parameters: {'num_leaves': 500, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 7, 'reg_alpha': 0.04002170011664538, 'reg_lambda': 0.6596717235718147}. Best is trial 51 with value: 0.909.
[I 2023-02-02 08:57:42,005] Trial 62 finished with value: 0.907 and parameters: {'num_leaves': 340, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 17, 'reg_alpha': 0.10179232787292555, 'reg_lambda': 0.7202289737778073}. Best is trial 51 with value: 0.909.
[I 2023-02-02 08:57:46,596] Trial 63 finished with value: 0.908 and parameters: {'num_leaves': 120, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 17, 'reg_alpha': 0.06036086182367619, 'reg_lambda': 0

[I 2023-02-02 08:59:23,973] Trial 90 finished with value: 0.903 and parameters: {'num_leaves': 1580, 'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.007453552797462186, 'reg_lambda': 0.6857873212441252}. Best is trial 85 with value: 0.91.
[I 2023-02-02 08:59:27,323] Trial 91 finished with value: 0.906 and parameters: {'num_leaves': 240, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 17, 'reg_alpha': 0.08890179615207311, 'reg_lambda': 0.7133719626510556}. Best is trial 85 with value: 0.91.
[I 2023-02-02 08:59:29,474] Trial 92 finished with value: 0.907 and parameters: {'num_leaves': 760, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 7, 'reg_alpha': 0.15394776961525064, 'reg_lambda': 0.8358210013036044}. Best is trial 85 with value: 0.91.
[I 2023-02-02 08:59:36,816] Trial 93 finished with value: 0.909 and parameters: {'num_leaves': 140, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.01680098792580772, 'reg_lambda':

[I 2023-02-02 09:01:40,304] Trial 120 finished with value: 0.908 and parameters: {'num_leaves': 520, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 17, 'reg_alpha': 0.04127063824453345, 'reg_lambda': 0.7066739231097463}. Best is trial 85 with value: 0.91.
[I 2023-02-02 09:01:49,328] Trial 121 finished with value: 0.908 and parameters: {'num_leaves': 580, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.014011070360445507, 'reg_lambda': 0.7662317591542919}. Best is trial 85 with value: 0.91.
[I 2023-02-02 09:01:57,054] Trial 122 finished with value: 0.908 and parameters: {'num_leaves': 320, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.010882474183498277, 'reg_lambda': 0.7773775965273628}. Best is trial 85 with value: 0.91.
[I 2023-02-02 09:02:04,646] Trial 123 finished with value: 0.907 and parameters: {'num_leaves': 360, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.011435830919116301, 'reg_la

...

[I 2023-02-02 09:57:15,504] Trial 900 finished with value: 0.907 and parameters: {'num_leaves': 2520, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.15313423785522437, 'reg_lambda': 0.7530692239875058}. Best is trial 410 with value: 0.911.
[I 2023-02-02 09:57:19,757] Trial 901 finished with value: 0.908 and parameters: {'num_leaves': 660, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 20, 'reg_alpha': 0.03663648452683264, 'reg_lambda': 0.6812238865645743}. Best is trial 410 with value: 0.911.
[I 2023-02-02 09:57:22,879] Trial 902 finished with value: 0.908 and parameters: {'num_leaves': 2740, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 20, 'reg_alpha': 0.13048266515842857, 'reg_lambda': 0.8771344425992755}. Best is trial 410 with value: 0.911.
[I 2023-02-02 09:57:28,323] Trial 903 finished with value: 0.522 and parameters: {'num_leaves': 2520, 'n_estimators': 500, 'learning_rate': 0.001, 'max_depth': 13, 'reg_alpha': 0.08184664229773944,

[I 2023-02-02 09:58:55,100] Trial 930 finished with value: 0.525 and parameters: {'num_leaves': 2880, 'n_estimators': 500, 'learning_rate': 0.001, 'max_depth': 13, 'reg_alpha': 0.037836146144899725, 'reg_lambda': 0.7767763938776835}. Best is trial 410 with value: 0.911.
[I 2023-02-02 09:58:55,365] Trial 931 finished with value: 0.736 and parameters: {'num_leaves': 2560, 'n_estimators': 10, 'learning_rate': 0.1, 'max_depth': 7, 'reg_alpha': 0.10934644962200313, 'reg_lambda': 0.7671716154172336}. Best is trial 410 with value: 0.911.
[I 2023-02-02 09:58:57,362] Trial 932 finished with value: 0.908 and parameters: {'num_leaves': 860, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 15, 'reg_alpha': 0.13659803240815505, 'reg_lambda': 0.7941552942556369}. Best is trial 410 with value: 0.911.
[I 2023-02-02 09:59:02,628] Trial 933 finished with value: 0.906 and parameters: {'num_leaves': 2400, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 11, 'reg_alpha': 0.02573484115655477, 

[I 2023-02-02 10:00:29,012] Trial 960 finished with value: 0.91 and parameters: {'num_leaves': 340, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.022808779590107416, 'reg_lambda': 0.8085066001385887}. Best is trial 410 with value: 0.911.
[I 2023-02-02 10:00:32,695] Trial 961 finished with value: 0.908 and parameters: {'num_leaves': 300, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.021853091763408477, 'reg_lambda': 0.6433274397587337}. Best is trial 410 with value: 0.911.
[I 2023-02-02 10:00:33,418] Trial 962 finished with value: 0.898 and parameters: {'num_leaves': 360, 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 20, 'reg_alpha': 0.21103935796409842, 'reg_lambda': 0.8280541512637628}. Best is trial 410 with value: 0.911.
[I 2023-02-02 10:00:35,500] Trial 963 finished with value: 0.908 and parameters: {'num_leaves': 520, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 17, 'reg_alpha': 0.09229100297493402, 're

[I 2023-02-02 10:01:57,742] Trial 990 finished with value: 0.908 and parameters: {'num_leaves': 2940, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.037648180640987364, 'reg_lambda': 0.7223694790631042}. Best is trial 410 with value: 0.911.
[I 2023-02-02 10:01:58,834] Trial 991 finished with value: 0.902 and parameters: {'num_leaves': 260, 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 17, 'reg_alpha': 0.040260921587902905, 'reg_lambda': 0.8085906840262918}. Best is trial 410 with value: 0.911.
[I 2023-02-02 10:02:04,564] Trial 992 finished with value: 0.908 and parameters: {'num_leaves': 2600, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.018575185899343392, 'reg_lambda': 0.7784660601185168}. Best is trial 410 with value: 0.911.
[I 2023-02-02 10:02:07,950] Trial 993 finished with value: 0.906 and parameters: {'num_leaves': 260, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.03176219275747062,

Number of finished trials: 1000


In [16]:
end = time.time()
print(end - start)

4084.9619057178497


In [17]:
print('Best trial:', study.best_trial.params)

Best trial: {'num_leaves': 840, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 20, 'reg_alpha': 0.044562263021639924, 'reg_lambda': 0.6630150015003023}


In [18]:

param = {
    "num_leaves": 840,
    "n_estimators": 500,
    'learning_rate': 0.1,
    'max_depth': 20,
    "reg_alpha": 0.044562263021639924,
    'reg_lambda':0.6630150015003023,
    
}
model = LGBMRegressor(**param)

model.fit(train_x,train_y,eval_set=[(test_x,test_y)], verbose=-1)

C:\Users\HuBEL\anaconda3\lib\site-packages\lightgbm\sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "


LGBMRegressor(max_depth=20, n_estimators=500, num_leaves=840,
              reg_alpha=0.044562263021639924, reg_lambda=0.6630150015003023)

In [19]:
yhat_train = model.predict(train_x)

predicted_train=pd.DataFrame(yhat_train,columns=['IAT'])
predicted_train = inverse_normalization_output(predicted_train)

actual_train = pd.DataFrame(train_y,columns=['IAT'])
actual_train = inverse_normalization_output(actual_train)

print('Training results')
print('r^2:',r2_score(actual_train, predicted_train))
print('RMSE:',np.sqrt(mean_squared_error(actual_train, predicted_train)))
print('MAE:',mean_absolute_error(actual_train, predicted_train))


Training results
r^2: 0.9987015432339159
RMSE: 0.07716183102417333
MAE: 0.049805311202628204


In [20]:
yhat_test = model.predict(test_x)
predicted_test=pd.DataFrame(yhat_test,columns=['IAT'])
predicted_test = inverse_normalization_output(predicted_test)

actual_test=pd.DataFrame(test_y,columns=['IAT'])
actual_test = inverse_normalization_output(actual_test)

print('Testing results')
print('r^2:',r2_score(actual_test, predicted_test))
print('RMSE:',np.sqrt(mean_squared_error(actual_test, predicted_test)))
print('MAE:',mean_absolute_error(actual_test, predicted_test))

Testing results
r^2: 0.9177040112129614
RMSE: 0.5877535820051973
MAE: 0.33909858665647413


In [21]:
pickle.dump(model, open('IAT.pkl','wb'))

In [22]:

train_df = pd.concat([actual_train, predicted_train], axis=1)
train_df.to_excel("training_results_past_2.xlsx",index=False)
test_df = pd.concat([actual_test, predicted_test], axis=1)
test_df.to_excel("testing_results_past_2.xlsx",index=False)